## Import raw data

In [4]:
from pathlib import Path
import polars as pl

kcat_df_raw = pl.read_csv("../data/cache/raw/kcat_train.csv")
km_df_raw = pl.read_csv("../data/cache/raw/km_train.csv")

## Get rid of unnecessary columns and rows with Null values

In [5]:
def clean_data(df):

    print('')

    print("Raw dataframe shape:", df.shape)

    print("Raw dataframe columns:", df.columns)

    idx = df.get_column_index('pdbpath')  # get the index of the 'pdbpath' column

    df = df.drop(df.columns[idx:]).drop_nulls()  # drop all columns from 'pdbpath' onwards and remove rows with null values

    print("Cleaned dataframe shape:", df.shape)

    print("Cleaned dataframe columns:", df.columns)

    print(df.head)

    return df

kcat_df = clean_data(kcat_df_raw)
km_df = clean_data(km_df_raw)


Raw dataframe shape: (18751, 26)
Raw dataframe columns: ['sequence', 'sequence_source', 'uniprot', 'reaction_smiles', 'value', 'reaction_mw_diff_perc', 'temperature', 'ph', 'ec', 'taxonomy_id', 'log10_value', 'reactant_smiles', 'product_smiles', 'log10kcat_max', 'group', 'pdbpath', 'reactant_smiles_20cluster', 'sequence_20cluster', 'reactant_smiles_40cluster', 'sequence_40cluster', 'reactant_smiles_60cluster', 'sequence_60cluster', 'reactant_smiles_80cluster', 'sequence_80cluster', 'reactant_smiles_99cluster', 'sequence_99cluster']
Cleaned dataframe shape: (15896, 15)
Cleaned dataframe columns: ['sequence', 'sequence_source', 'uniprot', 'reaction_smiles', 'value', 'reaction_mw_diff_perc', 'temperature', 'ph', 'ec', 'taxonomy_id', 'log10_value', 'reactant_smiles', 'product_smiles', 'log10kcat_max', 'group']
<bound method DataFrame.head of shape: (15_896, 15)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ sequence   ┆ sequence_ ┆ u

In [6]:
print(kcat_df.columns)
print(km_df.columns)

['sequence', 'sequence_source', 'uniprot', 'reaction_smiles', 'value', 'reaction_mw_diff_perc', 'temperature', 'ph', 'ec', 'taxonomy_id', 'log10_value', 'reactant_smiles', 'product_smiles', 'log10kcat_max', 'group']
['sequence', 'sequence_source', 'uniprot', 'substrate_smiles', 'value', 'temperature', 'ph', 'ec', 'taxonomy_id', 'log10_value', 'log10km_mean', 'group']


In [7]:
def count_duplicates(df, column):
    n_duplicates = df.select(pl.col(column).is_duplicated().sum()).item()
    print(f"Number of duplicate values in '{column}': {n_duplicates}")
    return n_duplicates

count_duplicates(kcat_df, "uniprot")
count_duplicates(km_df, "uniprot")

Number of duplicate values in 'uniprot': 12916
Number of duplicate values in 'uniprot': 26867


26867

In [8]:
kcat_df = kcat_df.rename({'reactant_smiles': 'substrate_smiles'})

In [9]:
def shared_reaction_params(_kcat_df, _km_df):

    _kcat_df = _kcat_df.with_columns(
        (pl.col('uniprot') + '_' + pl.col('substrate_smiles')).alias('uid_str')
    )

    _km_df = _km_df.with_columns(
        (pl.col('uniprot') + '_' + pl.col('substrate_smiles')).alias('uid_str')
    )

    # Keep only rows whose 'uid_str' appears exactly once
    # _km_df = _km_df.filter(pl.col('uid_str').is_unique())

    # print(_kcat_df.shape)
    # print(_kcat_df.head())

    # print(_km_df.shape)
    # print(_km_df.head())

    # # are there duplicates in the uid_str column
    # print(_kcat_df.select('uid_str').is_duplicated().sum())
    # print(_km_df.select('uid_str').is_duplicated().sum())

    # # Print all rows where 'uid_str' is duplicated
    # duplicated_rows = _km_df.filter(pl.col('uid_str').is_duplicated())
    # print(duplicated_rows)

    
    _km_df_shared = _km_df.filter(
        pl.col('uid_str').is_in(
            _kcat_df.select('uid_str').to_series().to_list()
        )
    )
    return _km_df_shared, _kcat_df.filter(
        pl.col('uid_str').is_in(
            _km_df_shared.select('uid_str').to_series().to_list()
        )
    )



In [10]:
df2 = km_df.filter(
    pl.col('uniprot').is_in(
        kcat_df.select('uniprot').to_series().to_list()
    ) & pl.col('substrate_smiles').is_in(
        kcat_df.select('substrate_smiles').to_series().to_list()
    )
)

df2.shape

(5508, 12)

In [11]:
kcat_df_shared, km_df_shared = shared_reaction_params(kcat_df, km_df)

In [12]:
print(kcat_df_shared)
print(km_df_shared)

shape: (1_337, 13)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ sequence   ┆ sequence_ ┆ uniprot ┆ substrate ┆ … ┆ log10_val ┆ log10km_m ┆ group     ┆ uid_str   │
│ ---        ┆ source    ┆ ---     ┆ _smiles   ┆   ┆ ue        ┆ ean       ┆ ---       ┆ ---       │
│ str        ┆ ---       ┆ str     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ str       ┆ str       │
│            ┆ str       ┆         ┆ str       ┆   ┆ f64       ┆ f64       ┆           ┆           │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ MASVTRAVFG ┆ brenda    ┆ Q96C23  ┆ OC[C@H]1O ┆ … ┆ 1.568202  ┆ 1.568202  ┆ OC[C@H]1O ┆ Q96C23_OC │
│ ELPSGGGTVE ┆           ┆         ┆ [C@H](O)[ ┆   ┆           ┆           ┆ [C@H](O)[ ┆ [C@H]1O[C │
│ KFQLQSDLLR ┆           ┆         ┆ C@H](O)[C ┆   ┆           ┆           ┆ C@H](O)[C ┆ @H](O)[C@ │
│ …          ┆           ┆         ┆ @@H…      ┆   ┆           ┆        